# Per-UID LSTM for IEEE-CIS Fraud — PyTorch version

This notebook mirrors `Time_Series_LSTM_per_UID.ipynb` (Keras) but is written in
PyTorch, following the structural pattern from the Kaggle reference:

> [arunmohan003 — *Sentiment analysis using LSTM - PyTorch*](https://www.kaggle.com/code/arunmohan003/sentiment-analysis-using-lstm-pytorch)

Same upstream logic as the Keras notebook:

- Load preprocessed checkpoints (`X_train_copy4.pkl`, `X_test_copy4.pkl`, `y_train.pkl`).
- Bridge train+test before windowing so test rows can use train history.
- Add three time-gap features per UID.
- Standardize (fit on train rows only).
- Build per-UID sliding windows of length `WINDOW`.
- Train under **strict expanding-window time validation** with `MIN_TRAIN_MONTHS = 3`.
- Save OOF and test predictions for ensembling with your XGBoost OOF.

## What's adapted from the sentiment-analysis reference

| arunmohan003's notebook | This notebook |
|--|--|
| Word indices `(batch, seq_len)` → `nn.Embedding` → `(batch, seq_len, embed_dim)` | Numeric features `(batch, seq_len, n_features)` directly into LSTM (no embedding) |
| `vocab_size`, `embedding_dim` hyperparameters | `n_features` only — no vocab |
| `nn.LSTM(input_size=embedding_dim, ...)` | `nn.LSTM(input_size=n_features, ...)` |
| `model.init_hidden(batch_size)` per iteration | Same pattern, kept for fidelity |
| `nn.BCELoss` after sigmoid | Same |
| Manual training loop with `optimizer.zero_grad()`, `loss.backward()`, `clip_grad_norm_`, `optimizer.step()` | Same |
| Best model saved by validation loss | Best model saved by **validation AUC** (better metric for fraud) |

The reason there's no embedding layer: in sentiment analysis each word is a discrete
token that has to be turned into a continuous vector. Your fraud features are already
continuous after `StandardScaler` (and previously-categorical fields like `card1` were
already integer-encoded by the upstream pipeline), so the LSTM can ingest them directly.


## MPS / Apple Silicon GPU notes

PyTorch supports Apple Silicon GPU via the **MPS** (Metal Performance Shaders) backend.
The config cell below picks the best available device automatically: `mps` if you're on
M-series, else `cuda`, else `cpu`.

A few specifics for MPS:

- Some ops fall back to CPU silently. For LSTM this works but you may see warnings
  about unsupported dtypes — mostly harmless.
- `torch.compile(...)` doesn't help much on MPS yet (Metal backend is limited),
  so this notebook doesn't use it.
- Mixed-precision (`autocast`) is supported but not used here — fp32 is more stable
  for LSTM on MPS, and the speed difference is small.
- `num_workers > 0` in `DataLoader` can deadlock on macOS in Jupyter. We use
  `num_workers=0` and rely on the unified-memory architecture for fast host-device
  transfer.


In [1]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/usr/bin/python3
None


In [2]:
# 0. Imports and config — PyTorch + MPS-aware
import os, gc, math, time, datetime, warnings, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# ----- Configuration -----
DATA_DIR = '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files'

WINDOW              = 20      # was 5  (change F)
MIN_TRAIN_MONTHS    = 3
BATCH               = 1024
EPOCHS              = 30      # was 12 (change I)
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0     # was 0.5 (change D)
EARLY_STOP_PATIENCE = 6
SEED                = 42

HIDDEN_DIM   = 128
NUM_LAYERS   = 2
DROPOUT      = 0.3
N_SEEDS      = 3              # for seed ensembling (change H)
USE_POS_WEIGHT = False        # plain BCE for AUC (change C)
USE_STATIC_TOWER = True       # dual-tower (change B)

# ----- Reproducibility -----
torch.manual_seed(SEED); np.random.seed(SEED)

# ----- Device selection -----
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'PyTorch {torch.__version__}  device={device}')


PyTorch 2.10.0+cu128  device=cuda


In [3]:
x = torch.randn(1024, 5, 250).to(device)
print(x.device)

cuda:0


## 1. Load preprocessed checkpoints (pickle)

In [4]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/bachhoviet/parquet-imported-files")

X_train = pd.read_parquet(DATA_DIR / "X_train_copy4.parquet")
X_test  = pd.read_parquet(DATA_DIR / "X_test_copy4.parquet")

y_train = pd.read_parquet(DATA_DIR / "y_train.parquet")["isFraud"].astype("int8")

In [5]:
X_train

,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,DT_M_total,DT_W_total,DT_D_total,uid_FE,uid_v2_FE,D1n,D2n,D3n,D5n,D9n
TransactionID,,,,,,,,,,,,,,,,,,,,,
2987000.0,86400.0,68.500000,4,12926.0,-1.0,50.0,1,42.0,1,215.0,...,137321,12093,5122,1.0,NaN,-13.0,2.0,-12.0,2.0,2.0
2987001.0,86401.0,29.000000,4,1755.0,304.0,50.0,2,2.0,1,225.0,...,137321,12093,5122,1.0,1.0,1.0,2.0,2.0,2.0,2.0
2987002.0,86469.0,59.000000,4,3663.0,390.0,50.0,3,66.0,2,230.0,...,137321,12093,5122,4.0,2.0,1.0,2.0,2.0,2.0,2.0
2987003.0,86499.0,50.000000,4,17132.0,467.0,50.0,2,17.0,2,376.0,...,137321,12093,5122,84.0,81.0,-111.0,-111.0,1.0,1.0,2.0
2987004.0,86506.0,50.000000,1,3497.0,414.0,50.0,2,2.0,1,320.0,...,137321,12093,5122,1.0,NaN,1.0,2.0,2.0,2.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3577535.0,15811047.0,49.000000,4,5550.0,-1.0,50.0,3,126.0,2,172.0,...,89326,10288,2754,2.0,2.0,153.0,153.0,152.0,183.0,183.0
3577536.0,15811049.0,39.500000,4,9444.0,125.0,50.0,2,124.0,2,104.0,...,89326,10288,2754,1.0,NaN,182.0,183.0,183.0,183.0,183.0
3577537.0,15811079.0,30.950001,4,11037.0,495.0,50.0,2,124.0,2,131.0,...,89326,10288,2754,9.0,6.0,182.0,183.0,183.0,183.0,183.0


In [6]:
X_test

,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,DT_M_total,DT_W_total,DT_D_total,uid_FE,uid_v2_FE,D1n,D2n,D3n,D5n,D9n
TransactionID,,,,,,,,,,,,,,,,,,,,,
3663549.0,18403224.0,31.950001,4,9409.0,11.0,50.0,3,126.0,2,70.0,...,78430,2243,2243,47.0,1.0,-206.0,-206.0,186.0,186.0,214.000000
3663550.0,18403264.0,49.000000,4,3272.0,11.0,50.0,3,126.0,2,199.0,...,78430,2243,2243,35.0,1.0,64.0,64.0,206.0,206.0,214.000000
3663551.0,18403310.0,171.000000,4,3476.0,474.0,50.0,3,126.0,2,372.0,...,78430,2243,2243,19.0,1.0,76.0,76.0,203.0,203.0,214.000000
3663552.0,18403310.0,284.950012,4,9989.0,260.0,50.0,3,66.0,2,105.0,...,78430,2243,2243,4.0,4.0,171.0,171.0,172.0,172.0,214.000000
3663553.0,18403316.0,67.949997,4,17018.0,352.0,50.0,2,17.0,2,164.0,...,78430,2243,2243,11.0,3.0,191.0,191.0,213.0,213.0,214.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4170235.0,34214280.0,94.679001,0,12832.0,275.0,85.0,2,124.0,2,184.0,...,116398,28121,2515,1.0,1.0,395.0,396.0,396.0,396.0,396.000000
4170236.0,34214288.0,12.173000,0,2154.0,308.0,85.0,2,124.0,2,-1.0,...,116398,28121,2515,38.0,2.0,379.0,379.0,379.0,379.0,396.000000
4170237.0,34214328.0,49.000000,4,15661.0,390.0,50.0,3,126.0,2,227.0,...,116398,28121,2515,4.0,1.0,395.0,396.0,396.0,396.0,396.000000


In [7]:
y_train

TransactionID
2987000.0    0
2987001.0    0
2987002.0    0
2987003.0    0
2987004.0    0
            ..
3577535.0    0
3577536.0    0
3577537.0    0
3577538.0    0
3577539.0    0
Name: isFraud, Length: 590540, dtype: int8

## 2. Feature selection


In [8]:
HOUSEKEEPING = {
    'TransactionDT', 'D6','D7','D8','D9','D12','D13','D14',
    'uid', 'uid_v2', 'uid_v2_FE', 'day', 'DT', 'isFraud',
    'C3','M5','id_08','id_33',
    'card4','id_07','id_14','id_21','id_30','id_32','id_34',
    *(f'id_{x}' for x in range(22, 28)),
    'D1n', 'D3n', 'D2n', 'D5n', 'D9n',
    'DT_M', 'DT_W', 'DT_D', 'DT_M_total', 'DT_W_total', 'DT_D_total'
}

feature_cols = [c for c in X_train.columns if c not in HOUSEKEEPING]
print('feature columns:', len(feature_cols))
print('Removed columns: ', len(HOUSEKEEPING))

feature columns: 224
Removed columns:  42


In [9]:
missing_housekeeping = sorted(HOUSEKEEPING - set(X_train.columns))
present_housekeeping = sorted(HOUSEKEEPING & set(X_train.columns))

print("HOUSEKEEPING total:", len(HOUSEKEEPING))
print("Present in X_train:", len(present_housekeeping))
print("Missing from X_train:", len(missing_housekeeping))
print(missing_housekeeping)

HOUSEKEEPING total: 42
Present in X_train: 42
Missing from X_train: 0
[]


In [10]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 224 FEATURES.


array(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5',
       'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain',
       'R_emaildomain', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
       'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5',
       'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4', 'M6', 'M7', 'M8',
       'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17',
       'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41',
       'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67',
       'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89',
       'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121',
       'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139',
       'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169',
       'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185',
       'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210',
       '

## 3. Bridge train+test, then add time-gap features

Same logic as the Keras notebook: concat first, compute gap features on the combined
frame so test rows of known UIDs can reference their train history.


In [11]:
X_train['__source__'] = 'train'
X_test['__source__']  = 'test'
X_all = pd.concat([X_train, X_test], axis=0)
# X_all.drop('isFraud', axis=1, inplace=True)
print(f'X_all rows: {len(X_all):,}')

def add_time_gap_features(df):
    df = df.sort_values(['uid', 'DT'], kind='mergesort').copy()
    g = df.groupby('uid', sort=False)

    delta = g['TransactionDT'].diff().fillna(0).astype('float32')
    df['delta_seconds_prev'] = delta
    # df['delta_log_prev']     = np.log1p(delta).astype('float32')
    df['uid_count_so_far']   = g.cumcount().astype('float32') + 1

    prev_amt = g['TransactionAmt'].shift(1)

    df['uid_prev_amt']       = prev_amt.fillna(-1).astype('float32')
    df['uid_amt_diff_prev']  = (df['TransactionAmt'] - prev_amt).fillna(0).astype('float32')
    df['uid_amt_ratio_prev'] = (
        df['TransactionAmt'] / prev_amt.replace(0, np.nan)
    ).fillna(1.0).clip(0, 100).astype('float32')

    df['uid_amt_cummean'] = g['TransactionAmt'].transform(
        lambda s: s.shift(1).expanding().mean()
    ).fillna(-1).astype('float32')

    df['uid_amt_cummax'] = g['TransactionAmt'].transform(
        lambda s: s.shift(1).expanding().max()
    ).fillna(-1).astype('float32')

    return df

X_all = add_time_gap_features(X_all)
feature_cols += [
    'delta_seconds_prev', 'uid_count_so_far',
    'uid_prev_amt', 'uid_amt_diff_prev', 'uid_amt_ratio_prev',
    'uid_amt_cummean', 'uid_amt_cummax'
    # , 'delta_log_prev'
]
print('feature columns now:', len(feature_cols))

# how many test rows actually benefit from the bridge?
X_all = X_all.sort_values(['uid', 'DT'], kind='mergesort')
prev_source = X_all.groupby('uid', sort=False)['__source__'].shift(1)
bridge_mask = (X_all['__source__'] == 'test') & (prev_source == 'train')
n_bridge = bridge_mask.sum()
test_total = (X_all['__source__'] == 'test').sum()
print(f'test rows whose immediate previous-UID-transaction is in TRAIN: '
      f'{n_bridge:,} ({n_bridge / test_total:.1%} of test rows)')


X_all rows: 1,097,231
feature columns now: 231
test rows whose immediate previous-UID-transaction is in TRAIN: 27,106 (5.3% of test rows)


In [12]:
print('X_all.index name :', X_all.index.name)             # should be 'TransactionID'
print('X_all.index[:3]  :', X_all.index[:3].tolist())     # should be 3 TransactionID-like ints
print('y_train.index[:3]:', y_train.index[:3].tolist())   # should match X_all's index
print('overlap:', X_all.index.isin(y_train.index).sum(),
      'of', len(y_train))                                 # should equal len(y_train) = 590540

X_all.index name : TransactionID
X_all.index[:3]  : [3230924.0, 3382464.0, 3493926.0]
y_train.index[:3]: [2987000.0, 2987001.0, 2987002.0]
overlap: 590540 of 590540


In [13]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 231 FEATURES.


array(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5',
       'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain',
       'R_emaildomain', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
       'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5',
       'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4', 'M6', 'M7', 'M8',
       'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17',
       'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41',
       'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67',
       'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89',
       'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121',
       'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139',
       'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169',
       'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185',
       'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210',
       '

In [14]:
cols_to_show = [
    "TransactionAmt",
    'uid_prev_amt', 'uid_amt_cummean', 'uid_amt_diff_prev', 'uid_amt_ratio_prev', 'uid_amt_cummax', "card1",
    "D1n", "uid", "uid_FE", 'uid_count_so_far', 'TransactionDT',  'delta_seconds_prev', "day", "D3", "D3n", "dist1", "P_emaildomain",
]

cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all.loc[X_all["uid"] == 48158, cols].head(20)

,TransactionAmt,uid_prev_amt,uid_amt_cummean,uid_amt_diff_prev,uid_amt_ratio_prev,uid_amt_cummax,card1,D1n,uid,uid_FE,uid_count_so_far,TransactionDT,delta_seconds_prev,day,D3,D3n,dist1,P_emaildomain
TransactionID,,,,,,,,,,,,,,,,,,
3428022.0,100.0,-1.0,-1.000000,0.0,1.000000,-1.0,14775.0,129.0,48158,1602.0,1.0,11204367.0,0.0,129.0,-1.0,130.0,-1.0,-1
3455679.0,10.0,100.0,100.000000,-90.0,0.100000,100.0,14775.0,129.0,48158,1602.0,2.0,12081968.0,877601.0,139.0,10.0,129.0,-1.0,-1
3465818.0,55.0,10.0,55.000000,45.0,5.500000,100.0,14775.0,129.0,48158,1602.0,3.0,12426792.0,344824.0,143.0,4.0,139.0,-1.0,-1
3470150.0,75.0,55.0,55.000000,20.0,1.363636,100.0,14775.0,129.0,48158,1602.0,4.0,12577656.0,150864.0,145.0,2.0,143.0,-1.0,-1
3473244.0,65.0,75.0,60.000000,-10.0,0.866667,100.0,14775.0,129.0,48158,1602.0,5.0,12673229.0,95573.0,146.0,1.0,145.0,-1.0,-1
3473999.0,55.0,65.0,61.000000,-10.0,0.846154,100.0,14775.0,129.0,48158,1602.0,6.0,12687363.0,14134.0,146.0,0.0,146.0,-1.0,-1
3475784.0,75.0,55.0,60.000000,20.0,1.363636,100.0,14775.0,129.0,48158,1602.0,7.0,12750993.0,63630.0,147.0,1.0,146.0,-1.0,-1
3476522.0,55.0,75.0,62.142857,-20.0,0.733333,100.0,14775.0,129.0,48158,1602.0,8.0,12766647.0,15654.0,147.0,0.0,147.0,-1.0,-1
3478472.0,65.0,55.0,61.250000,10.0,1.181818,100.0,14775.0,129.0,48158,1602.0,9.0,12835226.0,68579.0,148.0,1.0,147.0,-1.0,-1


In [15]:
cols_to_show = [
    "DT", "DT_M", "DT_W", "DT_D",
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday",
    "DT_M_total", "DT_W_total", "DT_D_total"
]

# keep only existing columns
cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all[cols].head(20)

,DT,DT_M,DT_W,DT_D,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday,DT_M_total,DT_W_total,DT_D_total
TransactionID,,,,,,,,,,,,,
3230924.0,2018-02-04 23:36:59,14,57,400,23,6,4,1,0,0,86021,23185,2584
3382464.0,2018-03-24 23:45:26,15,64,448,23,5,24,4,0,0,101632,21020,2758
3493926.0,2018-05-02 17:32:16,17,70,487,17,2,2,1,0,0,89326,22071,3105
3566032.0,2018-05-27 15:31:23,17,73,512,15,6,27,4,0,0,89326,19010,2068
3083895.0,2017-12-22 20:12:22,12,51,356,20,4,22,4,1,0,137321,41326,6852
3112200.0,2017-12-28 17:59:14,12,52,362,17,3,28,4,1,0,137321,26772,3332
3113013.0,2017-12-28 21:11:23,12,52,362,21,3,28,4,1,0,137321,26772,3332
3113097.0,2017-12-28 21:31:50,12,52,362,21,3,28,4,1,0,137321,26772,3332
4024997.0,2018-11-20 19:27:54,23,99,689,19,1,20,3,0,0,82804,17947,2740


In [16]:
X_all['uid'].value_counts()

uid
48158     1602
303723     587
285815     232
50510      230
277645     215
          ... 
387079       1
387081       1
387082       1
387084       1
387085       1
Name: count, Length: 387114, dtype: int64

## 4. Imputation + standardization (fit on train rows only)

In [17]:
RAW_CATEGORICALS = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']
RAW_CATEGORICALS = [c for c in RAW_CATEGORICALS if c in X_all.columns]
print(RAW_CATEGORICALS)

['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


In [18]:
fe_cols = [c for c in X_train.columns if c.endswith("_FE")]

print("Number of _FE columns:", len(fe_cols))
print(fe_cols)

Number of _FE columns: 9
['addr1_FE', 'card1_FE', 'card2_FE', 'card3_FE', 'P_emaildomain_FE', 'card1_addr1_FE', 'card1_addr1_P_emaildomain_FE', 'uid_FE', 'uid_v2_FE']


In [19]:
from collections import Counter

dupe_feature_cols = [c for c, n in Counter(feature_cols).items() if n > 1]
dupe_xall_cols = X_all.columns[X_all.columns.duplicated()].tolist()

print("duplicate feature_cols:", dupe_feature_cols)
print("duplicate X_all columns:", dupe_xall_cols)
feature_cols = list(dict.fromkeys(feature_cols))

duplicate feature_cols: []
duplicate X_all columns: []


In [20]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 231 FEATURES.


array(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5',
       'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain',
       'R_emaildomain', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
       'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5',
       'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4', 'M6', 'M7', 'M8',
       'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17',
       'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41',
       'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67',
       'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89',
       'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121',
       'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139',
       'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169',
       'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185',
       'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210',
       '

# ENCODING CATEGORICAL COLUMNS WITH THEIR FREQUENCIES

In [21]:
def cardinality(col):
    return X_all[col].nunique(dropna=False)

HIGH_CARD = [c for c in RAW_CATEGORICALS if cardinality(c) > 1000]   # FE only
MED_CARD  = [c for c in RAW_CATEGORICALS if 10 <= cardinality(c) <= 1000]  # FE + factorized
LOW_CARD  = [c for c in RAW_CATEGORICALS if cardinality(c) < 10]    # one-hot

In [22]:
print(HIGH_CARD)
print(MED_CARD)
print(LOW_CARD)

['DeviceInfo']
['P_emaildomain', 'R_emaildomain', 'id_30', 'id_31', 'id_33']
['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType']


In [23]:
low_card_raw_cats = [
    c for c in RAW_CATEGORICALS
    if c in X_all.columns and X_all[c].nunique(dropna=True) <= 3
]

np.array(low_card_raw_cats)

array(['M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_16',
       'id_27', 'id_28', 'id_29', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType'], dtype='<U10')

In [24]:
X_all.card4.value_counts()

card4
 3    719649
 2    347386
 0     16009
 1      9524
-1      4663
Name: count, dtype: int64

In [25]:
# FREQUENCY ENCODE TOGETHER
def encode_FE(df, cols):
    for col in cols:
        vc = df[col].value_counts(dropna=True, normalize=False).to_dict()
        vc[-1] = -1
        nm = col+'_FE'
        df[nm] = df[col].map(vc).astype('float32')
        print(nm,', ',end='')

In [26]:
FE_RAW_CATS = [c for c in RAW_CATEGORICALS if c not in low_card_raw_cats]
encode_FE(X_all, FE_RAW_CATS)

ProductCD_FE , card4_FE , card6_FE , P_emaildomain_FE , R_emaildomain_FE , M4_FE , id_15_FE , id_23_FE , id_30_FE , id_31_FE , id_33_FE , id_34_FE , DeviceInfo_FE , 

In [27]:
X_all.drop(columns=FE_RAW_CATS, inplace=True)

In [28]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 231 FEATURES.


array(['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5',
       'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain',
       'R_emaildomain', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
       'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5',
       'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4', 'M6', 'M7', 'M8',
       'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17',
       'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41',
       'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67',
       'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89',
       'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121',
       'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139',
       'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169',
       'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185',
       'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210',
       '

In [29]:
replace_raw_cats = [c for c in FE_RAW_CATS if c in feature_cols]

feature_cols = [
    f"{c}_FE" if c in replace_raw_cats else c
    for c in feature_cols
]

In [30]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 231 FEATURES.


array(['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3',
       'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2',
       'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5',
       'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1',
       'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3',
       'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8',
       'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30',
       'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56',
       'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80',
       'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115',
       'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130',
       'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162',
       'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178',
       'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205',
       'V207', 'V

In [31]:
feature_cols = list(dict.fromkeys(feature_cols))

In [32]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 230 FEATURES.


array(['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3',
       'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2',
       'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5',
       'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1',
       'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3',
       'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8',
       'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30',
       'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56',
       'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80',
       'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115',
       'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130',
       'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162',
       'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178',
       'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205',
       'V207', 'V

In [33]:
def add_cyclical_features(df, drop_originals=True):
    """Replace cyclical integer columns with sin/cos pairs in [-1, 1]."""
    cyclical = {
        'DT_hour':       24,
        'DT_day_week':   7,
        'DT_day_month':  31,
        'DT_week_month': 5,
    }
    for col, period in cyclical.items():
        if col not in df.columns:
            continue
        rad = 2 * np.pi * df[col].astype('float32') / period
        df[f'{col}_sin'] = np.sin(rad).astype('float32')
        df[f'{col}_cos'] = np.cos(rad).astype('float32')
    if drop_originals:
        df = df.drop(columns=[c for c in cyclical if c in df.columns])
    return df

In [34]:
cols_to_show = [
    "DT_hour", "DT_day_week", "DT_day_month", "DT_week_month",
    "is_december", "is_holiday"
]

# keep only existing columns
cols = [c for c in cols_to_show if c in X_all.columns]
missing = [c for c in cols_to_show if c not in X_all.columns]

if missing:
    print("Missing columns:", missing)

X_all[cols].head(20)

,DT_hour,DT_day_week,DT_day_month,DT_week_month,is_december,is_holiday
TransactionID,,,,,,
3230924.0,23,6,4,1,0,0
3382464.0,23,5,24,4,0,0
3493926.0,17,2,2,1,0,0
3566032.0,15,6,27,4,0,0
3083895.0,20,4,22,4,1,0
3112200.0,17,3,28,4,1,0
3113013.0,21,3,28,4,1,0
3113097.0,21,3,28,4,1,0
4024997.0,19,1,20,3,0,0


In [35]:
X_all = add_cyclical_features(X_all)

# Update feature_cols accordingly
for c in ['DT_hour', 'DT_day_week', 'DT_day_month', 'DT_week_month']:
    if c in feature_cols:
        feature_cols.remove(c)
    feature_cols += [f'{c}_sin', f'{c}_cos']

In [36]:
sin_cos_cols = [c for c in X_all.columns if c.endswith(("_sin", "_cos"))]

print(len(sin_cos_cols))
print(sin_cos_cols)

8
['DT_hour_sin', 'DT_hour_cos', 'DT_day_week_sin', 'DT_day_week_cos', 'DT_day_month_sin', 'DT_day_month_cos', 'DT_week_month_sin', 'DT_week_month_cos']


In [37]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 234 FEATURES.


array(['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3',
       'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2',
       'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5',
       'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1',
       'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3',
       'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8',
       'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30',
       'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56',
       'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80',
       'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115',
       'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130',
       'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162',
       'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178',
       'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205',
       'V207', 'V

In [38]:
train_mask = X_all["__source__"].eq("train")

candidate_cols = [
    c for c in feature_cols
    if pd.api.types.is_numeric_dtype(X_all[c])
]

def skew_report(df, cols, sentinel=-1):
    rows = []

    for c in cols:
        s = df[c].replace([np.inf, -np.inf], np.nan).dropna()

        # Ignore sentinel when judging skew
        s_valid = s[s != sentinel]

        if len(s_valid) < 100:
            continue

        rows.append({
            "col": c,
            "min": s_valid.min(),
            "p50": s_valid.quantile(0.50),
            "p95": s_valid.quantile(0.95),
            "p99": s_valid.quantile(0.99),
            "max": s_valid.max(),
            "skew": s_valid.skew(),
            "zero_rate": (s_valid == 0).mean(),
        })

    return (
        pd.DataFrame(rows)
        .sort_values("skew", ascending=False)
        .reset_index(drop=True)
    )

In [39]:
report = skew_report(X_all.loc[train_mask], candidate_cols)
report

,col,min,p50,p95,p99,max,skew,zero_rate
0,V305,0.0,0.0,0.000000,0.0,1.0,384.226166,0.999993
1,V129,0.0,0.0,29.000000,214.0,55125.0,240.274261,0.946454
2,V309,0.0,0.0,54.500000,226.0,55125.0,224.875259,0.923838
3,V266,0.0,0.0,28.656639,150.0,55125.0,175.958771,0.923982
4,M1,0.0,0.0,0.000000,0.0,1.0,113.025307,0.999922
...,...,...,...,...,...,...,...,...
229,V41,0.0,1.0,1.000000,1.0,1.0,-36.956020,0.000731
230,V14,0.0,1.0,1.000000,1.0,1.0,-44.708271,0.000500
231,V107,0.0,1.0,1.000000,1.0,1.0,-48.754040,0.000420
232,V65,0.0,1.0,1.000000,1.0,1.0,-54.450912,0.000337


In [40]:
log1p_candidate_cols = (
    report
    .loc[
        (report["skew"] > 2) &
        (report["min"] >= 0),
        "col"
    ]
    .tolist()
)

np.array(log1p_candidate_cols)

array(['V305', 'V129', 'V309', 'V266', 'M1', 'V205', 'V240', 'V320',
       'V277', 'V221', 'V291', 'V136', 'V68', 'V27', 'V253', 'V89',
       'V267', 'V111', 'V226', 'V224', 'V229', 'V44', 'C12', 'V166', 'C7',
       'V271', 'V252', 'V130', 'C8', 'V127', 'V215', 'C10', 'V162',
       'V264', 'C1', 'V294', 'C2', 'V307', 'V56', 'uid_count_so_far',
       'V37', 'V138', 'C11', 'C4', 'uid_prev_amt', 'V207', 'V261', 'V274',
       'C6', 'V218', 'V310', 'uid_amt_ratio_prev', 'V296', 'V108',
       'uid_FE', 'V258', 'V86', 'C14', 'V187', 'uid_amt_cummax', 'V198',
       'V78', 'uid_amt_cummean', 'V210', 'TransactionAmt', 'dollars',
       'V121', 'V257', 'V178', 'V297', 'V188', 'V123', 'V120', 'V283',
       'V285', 'V176', 'V23', 'V209', 'V223', 'V235', 'V228', 'V314',
       'V203', 'V301', 'V220', 'V238', 'V47', 'V281', 'V142', 'V147',
       'V185', 'V117', 'V182', 'C13', 'V169', 'V180', 'V8', 'V156',
       'TransactionAmt_card1_addr1_P_emaildomain_mean', 'V286',
       'TransactionAmt

In [41]:
print(len(log1p_candidate_cols))

140


In [42]:
fe_cols = [c for c in feature_cols if c.endswith("_FE")]

print("Number of _FE columns:", len(fe_cols))
print(fe_cols)

Number of _FE columns: 15
['ProductCD_FE', 'card6_FE', 'P_emaildomain_FE', 'R_emaildomain_FE', 'M4_FE', 'id_15_FE', 'id_31_FE', 'DeviceInfo_FE', 'addr1_FE', 'card1_FE', 'card2_FE', 'card3_FE', 'card1_addr1_FE', 'card1_addr1_P_emaildomain_FE', 'uid_FE']


In [43]:
log1p_cols = [
    c for c in log1p_candidate_cols
    if c in X_all.columns
    and c not in sin_cos_cols
    and X_all[c].nunique(dropna=True) > 3
]

np.array(log1p_cols)

array(['V129', 'V309', 'V266', 'V205', 'V240', 'V320', 'V277', 'V221',
       'V291', 'V136', 'V68', 'V27', 'V253', 'V89', 'V267', 'V111',
       'V226', 'V224', 'V229', 'V44', 'C12', 'V166', 'C7', 'V271', 'V252',
       'V130', 'C8', 'V127', 'V215', 'C10', 'V162', 'V264', 'C1', 'V294',
       'C2', 'V307', 'V56', 'uid_count_so_far', 'V37', 'V138', 'C11',
       'C4', 'uid_prev_amt', 'V207', 'V261', 'V274', 'C6', 'V218', 'V310',
       'uid_amt_ratio_prev', 'V296', 'V108', 'uid_FE', 'V258', 'V86',
       'C14', 'V187', 'uid_amt_cummax', 'V198', 'V78', 'uid_amt_cummean',
       'V210', 'TransactionAmt', 'dollars', 'V121', 'V257', 'V178',
       'V297', 'V188', 'V123', 'V120', 'V283', 'V285', 'V176', 'V23',
       'V209', 'V223', 'V235', 'V228', 'V314', 'V203', 'V301', 'V220',
       'V238', 'V47', 'V281', 'V142', 'V147', 'V185', 'V117', 'V182',
       'C13', 'V169', 'V180', 'V8', 'V156',
       'TransactionAmt_card1_addr1_P_emaildomain_mean', 'V286',
       'TransactionAmt_card1_addr1_P

In [44]:
print(len(log1p_cols))

135


In [45]:
def signed_log1p(x):
    """For features that can go negative (uid_amt_diff_prev)."""
    return np.sign(x) * np.log1p(np.abs(x))

In [46]:
def safe_log1p_with_sentinel(s, sentinel=-1):
    out = s.astype('float32').copy()
    valid = s.ne(sentinel)

    if (s.loc[valid] < 0).any():
        raise ValueError("Non-sentinel negative values found before log1p transform")

    out.loc[valid] = np.log1p(s.loc[valid]).astype('float32')
    return out

In [47]:
cols_to_log1p = list(dict.fromkeys(log1p_cols))
print(np.array(cols_to_log1p))
print(len(cols_to_log1p))

['V129' 'V309' 'V266' 'V205' 'V240' 'V320' 'V277' 'V221' 'V291' 'V136'
 'V68' 'V27' 'V253' 'V89' 'V267' 'V111' 'V226' 'V224' 'V229' 'V44' 'C12'
 'V166' 'C7' 'V271' 'V252' 'V130' 'C8' 'V127' 'V215' 'C10' 'V162' 'V264'
 'C1' 'V294' 'C2' 'V307' 'V56' 'uid_count_so_far' 'V37' 'V138' 'C11' 'C4'
 'uid_prev_amt' 'V207' 'V261' 'V274' 'C6' 'V218' 'V310'
 'uid_amt_ratio_prev' 'V296' 'V108' 'uid_FE' 'V258' 'V86' 'C14' 'V187'
 'uid_amt_cummax' 'V198' 'V78' 'uid_amt_cummean' 'V210' 'TransactionAmt'
 'dollars' 'V121' 'V257' 'V178' 'V297' 'V188' 'V123' 'V120' 'V283' 'V285'
 'V176' 'V23' 'V209' 'V223' 'V235' 'V228' 'V314' 'V203' 'V301' 'V220'
 'V238' 'V47' 'V281' 'V142' 'V147' 'V185' 'V117' 'V182' 'C13' 'V169'
 'V180' 'V8' 'V156' 'TransactionAmt_card1_addr1_P_emaildomain_mean' 'V286'
 'TransactionAmt_card1_addr1_P_emaildomain_std' 'V165'
 'TransactionAmt_card1_addr1_mean' 'V171' 'V6' 'V40' 'V115' 'V175' 'V234'
 'V173' 'V124' 'dist2' 'V139' 'C5' 'C9' 'V80' 'V3'
 'TransactionAmt_card1_addr1_std' 'V284' 

In [48]:
def has_real_negatives(s, sentinel=-1):
    """True if column has negatives other than the sentinel."""
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    s_valid = s[s != sentinel]
    return (s_valid < 0).any()

negatives = [c for c in cols_to_log1p if has_real_negatives(X_all[c])]
print(f'Columns with non-sentinel negatives: {len(negatives)}')
print(negatives)

Columns with non-sentinel negatives: 0
[]


In [49]:
for c in cols_to_log1p:
    X_all[c] = safe_log1p_with_sentinel(X_all[c])

In [50]:
def signed_log1p_candidates(df, cols, sentinel=-1, min_negatives=100, min_skew=2):
    out = []
    for c in cols:
        s = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        s_valid = s[s != sentinel]
        n_neg = (s_valid < 0).sum()
        n_pos = (s_valid > 0).sum()
        if n_neg < min_negatives or n_pos < min_negatives:
            continue
        if abs(s_valid.skew()) < min_skew and abs(s_valid.kurt()) < 5:
            continue
        out.append({'col': c, 'n_neg': int(n_neg), 'n_pos': int(n_pos),
                    'min': s_valid.min(), 'max': s_valid.max(),
                    'skew': s_valid.skew()})
    return pd.DataFrame(out).sort_values('skew', key=abs, ascending=False)

print(signed_log1p_candidates(X_all, feature_cols))

                 col   n_neg   n_pos      min      max      skew
0  uid_amt_diff_prev  259767  248935 -30515.0  30515.0  0.321907


In [51]:
X_all['uid_amt_diff_prev'] = signed_log1p(X_all['uid_amt_diff_prev']).astype('float32')

In [52]:
print('NOW USING THE FOLLOWING',len(feature_cols),'FEATURES.')
np.array(feature_cols)

NOW USING THE FOLLOWING 234 FEATURES.


array(['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3',
       'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2',
       'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5',
       'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1',
       'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3',
       'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8',
       'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30',
       'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56',
       'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80',
       'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115',
       'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130',
       'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162',
       'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178',
       'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205',
       'V207', 'V

In [53]:
print(X_all.isna().sum()[X_all.isna().sum() > 0])

isFraud      506691
uid_v2       363333
uid_v2_FE    363333
dtype: int64


In [54]:
# X_all[feature_cols] = X_all[feature_cols].fillna(-1).astype('float32')

In [55]:
train_mask = (X_all['__source__'] == 'train')
scaler = StandardScaler()

binary_cols = [
    c for c in feature_cols
    if c in X_all.columns and X_all[c].nunique(dropna=True) <= 3
]

to_scale = [
    c for c in feature_cols
    if c not in binary_cols
    and c not in sin_cos_cols
]

In [56]:
np.array(binary_cols)

array(['M1', 'M2', 'M3', 'M6', 'M7', 'M8', 'M9', 'V1', 'V14', 'V41',
       'V65', 'V88', 'V107', 'V305', 'id_12', 'id_16', 'id_28', 'id_29',
       'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'is_december',
       'is_holiday', 'DT_week_month_cos'], dtype='<U17')

In [57]:
np.array(sin_cos_cols)

array(['DT_hour_sin', 'DT_hour_cos', 'DT_day_week_sin', 'DT_day_week_cos',
       'DT_day_month_sin', 'DT_day_month_cos', 'DT_week_month_sin',
       'DT_week_month_cos'], dtype='<U17')

In [58]:
print(to_scale)
print(len(to_scale))

['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3', 'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M4_FE', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V89', 'V91', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210', 'V215', 'V218', 'V220', 'V221', 'V223', 'V224', 'V226', 'V228', 'V229', 'V234', 'V235', 'V238', 'V240', 'V250', 'V252', 'V253', 'V257', 'V258', 'V260', 'V261', 'V264', 'V266'

In [59]:
print(feature_cols)
print(len(feature_cols))

['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3', 'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210', 'V215', 'V218', 'V220', 'V221', 'V223', 'V224', 'V226', 'V228', 'V229', 'V234', 'V235', 'V

In [60]:
print(X_all[to_scale].dtypes[X_all[to_scale].dtypes != "float32"])

card1_addr1                  int32
card1_addr1_P_emaildomain    int32
dtype: object


In [61]:
print('uid_amt_diff_prev range:', X_all['uid_amt_diff_prev'].min(),
                                  X_all['uid_amt_diff_prev'].max())
# After ONLY signed_log1p (before this cell ran): roughly [-12, 12]
# If you see anything like ±200, signed_log1p never ran — that's the original bug

uid_amt_diff_prev range: -10.326006889343262 10.326006889343262


In [62]:
# 1. The 5 first features should be roughly N(0, 1) — but slightly tighter due to ±5 clip
sub = X_all.loc[train_mask, to_scale]
print('before-scale stats over to_scale:')
print('  min  :', float(sub.min().min()))   # should be exactly -5.0 (or close)
print('  max  :', float(sub.max().max()))   # should be exactly +5.0 (or close)
print('  mean :', float(sub.mean().mean())) # should be ~0
print('  std  :', float(sub.std().mean()))  # should be ~0.7 - 0.95

# 2. The 33 NOT-scaled columns should still live in their natural ranges
not_scaled = [c for c in feature_cols if c not in to_scale]
sub2 = X_all[not_scaled]
print('\nnot_scaled columns range:')
print('  min :', float(sub2.min().min()))   # expect -1 (TF_NULL sentinel) or -1 (cyclic)
print('  max :', float(sub2.max().max()))   # expect +1 (TF_NULL or cyclic) or near 1

# 3. No NaN/Inf anywhere
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
assert np.isfinite(flat).all(), 'NaN/Inf still present'
print('\n✓ all finite')

before-scale stats over to_scale:
  min  : -10.326006889343262
  max  : 999594.0
  mean : 13876.487260232907
  std  : 7612.702012190429

not_scaled columns range:
  min : -1.0
  max : 1.0

✓ all finite


In [63]:
feature_cols = list(dict.fromkeys(feature_cols))
to_scale = list(dict.fromkeys(to_scale))

X_all[feature_cols] = (
    X_all[feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(-1)
    .astype("float32")
)

train_mask = X_all["__source__"].eq("train")
scaler = StandardScaler()

X_all.loc[train_mask, to_scale] = scaler.fit_transform(
    X_all.loc[train_mask, to_scale]
).astype("float32")

X_all.loc[~train_mask, to_scale] = scaler.transform(
    X_all.loc[~train_mask, to_scale]
).astype("float32")

X_all[to_scale] = X_all[to_scale].clip(lower=-5.0, upper=5.0).astype("float32")

print('train post-scale mean (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].mean().round(3).to_list())
print('train post-scale std  (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].std().round(3).to_list())

train post-scale mean (first 5): [-0.0, 0.0, -0.0, 0.0, 0.03200000151991844]
train post-scale std  (first 5): [1.0, 0.9990000128746033, 1.0, 1.0, 0.6420000195503235]


In [64]:
print('uid_amt_diff_prev range:', X_all['uid_amt_diff_prev'].min(),
                                  X_all['uid_amt_diff_prev'].max())
# After signed_log1p AND scaling+clip, should be in [-5, 5]
# If you see anything like ±200, signed_log1p never ran — that's the original bug

uid_amt_diff_prev range: -4.005455493927002 4.035607814788818


In [65]:
# 1. The 5 first features should be roughly N(0, 1) — but slightly tighter due to ±5 clip
sub = X_all.loc[train_mask, to_scale]
print('post-scale stats over to_scale:')
print('  min  :', float(sub.min().min()))   # should be exactly -5.0 (or close)
print('  max  :', float(sub.max().max()))   # should be exactly +5.0 (or close)
print('  mean :', float(sub.mean().mean())) # should be ~0
print('  std  :', float(sub.std().mean()))  # should be ~0.7 - 0.95

# 2. The 33 NOT-scaled columns should still live in their natural ranges
not_scaled = [c for c in feature_cols if c not in to_scale]
sub2 = X_all[not_scaled]
print('\nnot_scaled columns range:')
print('  min :', float(sub2.min().min()))   # expect -1 (TF_NULL sentinel) or -1 (cyclic)
print('  max :', float(sub2.max().max()))   # expect +1 (TF_NULL or cyclic) or near 1

# 3. No NaN/Inf anywhere
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
assert np.isfinite(flat).all(), 'NaN/Inf still present'
print('\n✓ all finite')

post-scale stats over to_scale:
  min  : -5.0
  max  : 5.0
  mean : -0.0032160175032913685
  std  : 0.9518962502479553

not_scaled columns range:
  min : -1.0
  max : 1.0

✓ all finite


In [66]:
feature_cols[:5]

['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3']

In [67]:
[(c, c in to_scale) for c in feature_cols[:5]]

[('TransactionAmt', True),
 ('ProductCD_FE', True),
 ('card1', True),
 ('card2', True),
 ('card3', True)]

In [68]:
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
print('min:', flat.min(),  '   should be ≥ -5')
print('max:', flat.max(),  '   should be ≤ +5')
assert np.isfinite(flat).all()

min: -5.0    should be ≥ -5
max: 5.0    should be ≤ +5


In [69]:
flat = X_all[feature_cols].to_numpy(dtype=np.float32)
print('min :', flat.min(),  '  expected ~ -5')
print('max :', flat.max(),  '  expected ~ +5')
print('mean:', flat.mean(), '  expected ~ 0')
print('std :', flat.std(),  '  expected ~ 0.6 - 1.0')
assert np.isfinite(flat).all(), 'still has NaN/Inf'

min : -5.0   expected ~ -5
max : 5.0   expected ~ +5
mean: 0.010609204   expected ~ 0
std : 0.9451525   expected ~ 0.6 - 1.0


## 5. Build per-UID current-transaction windows

UPDATED v6: this variant keeps the UID-disjoint split from v4, but changes the
supervision target to **one scalar label per current transaction**.

Each sample is one transaction represented by that UID's history up to and
including the current transaction, right-padded to `MAX_LEN`. The model output is
one fraud logit per sample, not one logit per timestep. There are no padding
labels in v6 because each sequence has exactly one scalar target.


In [70]:
def split_uids_by_row_count(uid_series, frac_train=0.8, seed=42):
    rng = np.random.default_rng(seed)
    uid_counts  = uid_series.value_counts()
    uids_arr    = uid_counts.index.to_numpy()
    counts_arr  = uid_counts.values.astype(np.int64)

    perm        = rng.permutation(len(uids_arr))
    uids_shuf   = uids_arr[perm]
    counts_shuf = counts_arr[perm]

    total       = int(counts_shuf.sum())
    target      = int(frac_train * total)
    cumsum      = np.cumsum(counts_shuf)
    cut         = int(np.searchsorted(cumsum, target)) + 1

    return (set(uids_shuf[:cut].tolist()),
            set(uids_shuf[cut:].tolist()),
            total,
            int(counts_shuf[:cut].sum()),
            int(counts_shuf[cut:].sum()))


In [71]:
def build_current_txn_windows(df, feature_cols, max_len,
                              uid_col='uid', time_col='DT', y_col='_y'):
    """One sample per transaction: right-padded UID history ending at that row."""
    df_sorted = df.sort_values([uid_col, time_col], kind='mergesort')
    feat = df_sorted[feature_cols].to_numpy(dtype=np.float32)
    y = df_sorted[y_col].to_numpy(dtype=np.float32)
    uids = df_sorted[uid_col].to_numpy()
    orig = df_sorted.index.to_numpy()

    n = len(df_sorted)
    F = len(feature_cols)
    X = np.zeros((n, max_len, F), dtype=np.float32)
    L = np.zeros(n, dtype=np.int64)
    y_out = y.copy()
    orig_out = orig.copy()
    uid_out = uids.copy()

    boundaries = np.flatnonzero(np.concatenate([[True], uids[1:] != uids[:-1]]))
    boundaries = np.append(boundaries, n)

    for b_start, b_end in zip(boundaries[:-1], boundaries[1:]):
        block = feat[b_start:b_end]
        for t in range(b_end - b_start):
            row = b_start + t
            start = max(0, t - max_len + 1)
            seq = block[start:t + 1]
            X[row, :seq.shape[0], :] = seq
            L[row] = seq.shape[0]

    return X, L, y_out, orig_out, uid_out


In [72]:
# --- 5a. Prepare the train dataframe sorted by (uid, DT) with labels attached ---
train_mask = X_all['__source__'].eq('train')
df_train = X_all.loc[train_mask, feature_cols + ['uid', 'DT']].copy()
df_train['_y'] = y_train.reindex(df_train.index).to_numpy().astype(np.float32)
df_train = df_train.sort_values(['uid', 'DT'], kind='mergesort')

# --- 5b. Decide MAX_LEN. p99 cap keeps memory sane vs the long tail. ---
uid_len = df_train.groupby('uid', sort=False).size()
print('UID length distribution:')
print(f'  min={uid_len.min()}  median={uid_len.median():.0f}  '
      f'mean={uid_len.mean():.1f}  p95={uid_len.quantile(0.95):.0f}  '
      f'p99={uid_len.quantile(0.99):.0f}  max={uid_len.max()}')

MAX_LEN = int(min(uid_len.max(), uid_len.quantile(0.99)))
n_history_capped = int((uid_len > MAX_LEN).sum())
print(f'\nMAX_LEN = {MAX_LEN}   (history capped for {n_history_capped:,} long UIDs)')

# --- 5c. UID split by transaction count ~80/20 ---
train_uid_set, val_uid_set, total_rows, n_tr, n_va = split_uids_by_row_count(
    df_train['uid'], frac_train=0.8, seed=42,
)
assert train_uid_set.isdisjoint(val_uid_set)

print(f'\nTotal train transactions: {total_rows:,}')
print(f'  Train: {len(train_uid_set):>7,} UIDs, {n_tr:>7,} rows ({n_tr/total_rows:.1%})')
print(f'  Val  : {len(val_uid_set):>7,} UIDs, {n_va:>7,} rows ({n_va/total_rows:.1%})')

# --- 5d. Build current-transaction windows separately for train/val UID sets ---
df_tr = df_train.loc[df_train['uid'].isin(train_uid_set)].copy()
df_va = df_train.loc[df_train['uid'].isin(val_uid_set)].copy()

print('\nBuilding current-transaction windows...')
t0 = time.time()
X_tr, L_tr, y_tr, orig_tr, uids_tr = build_current_txn_windows(
    df_tr, feature_cols, MAX_LEN
)
X_va, L_va, y_va, orig_va, uids_va = build_current_txn_windows(
    df_va, feature_cols, MAX_LEN
)

print(f'  train shape={X_tr.shape}   val shape={X_va.shape}   in {time.time()-t0:.1f}s')
print(f'  train tensor={X_tr.nbytes/1e9:.2f} GB   val tensor={X_va.nbytes/1e9:.2f} GB')
print(f'  mean real-steps train={L_tr.mean():.2f}   val={L_va.mean():.2f}')
print(f'  pct train with full MAX_LEN history: {(L_tr == MAX_LEN).mean():.1%}')
print(f'  pct val with full MAX_LEN history:   {(L_va == MAX_LEN).mean():.1%}')

assert len(X_tr) == len(y_tr) == len(orig_tr)
assert len(X_va) == len(y_va) == len(orig_va)
assert set(pd.unique(uids_tr)).isdisjoint(set(pd.unique(uids_va)))

gc.collect()


UID length distribution:
  min=1  median=1  mean=2.7  p95=9  p99=20  max=1414

MAX_LEN = 20   (history capped for 2,176 long UIDs)

Total train transactions: 590,540
  Train: 175,043 UIDs, 472,434 rows (80.0%)
  Val  :  42,692 UIDs, 118,106 rows (20.0%)

Building current-transaction windows...
  train shape=(472434, 20, 234)   val shape=(118106, 20, 234)   in 5.3s
  train tensor=8.84 GB   val tensor=2.21 GB
  mean real-steps train=4.59   val=4.85
  pct train with full MAX_LEN history: 5.2%
  pct val with full MAX_LEN history:   6.5%


0

## 6. PyTorch `Dataset` and `DataLoader`

UPDATED v6: each dataset row is one current transaction window plus one scalar
fraud label. No timestep label mask is needed in this version.


In [73]:
class CurrentTxnWindowDataset(Dataset):
    """One sample = one current transaction with UID history right-padded to MAX_LEN."""
    def __init__(self, X, L, y):
        self.X = torch.from_numpy(X)
        self.L = torch.from_numpy(L.astype('int64'))
        self.y = torch.from_numpy(y.astype('float32'))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.L[i], self.y[i]


def make_loader(X, L, y, batch_size, shuffle, device=None):
    return DataLoader(
        CurrentTxnWindowDataset(X, L, y),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=False,
        drop_last=False,
    )


## 7. PyTorch LSTM model

UPDATED v6: the model predicts only the **current transaction** for each UID
history window. It uses `pack_padded_sequence` on right-padded windows, then
combines the last real hidden state with mean+max pooling over the available
history. Because each sequence ends at the current transaction, this pooling does
not look into future transactions.


In [76]:
class FraudLSTMCurrentTxn(nn.Module):
    """Scalar-output LSTM for the current transaction in a UID history window."""
    def __init__(self, n_features, hidden_dim=128, num_layers=2,
                 drop_prob=0.3, bidirectional=False):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=drop_prob if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)

        # Last real hidden state + history mean pool + history max pool.
        self.static_mlp = nn.Sequential(
            nn.Linear(n_features, 256), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(256, 128),        nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(128, 64),         nn.ReLU(),
        )
        head_in = out_dim * 3 + 64       # was out_dim * 3
        
        self.head = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(), nn.Dropout(drop_prob),
            nn.Linear(64, 1),
        )

    def forward(self, x, lengths):
        B, T, _ = x.shape
        lengths_dev = lengths.to(x.device)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.detach().cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=T
        )

        idx = torch.arange(T, device=x.device).unsqueeze(0)
        mask = idx < lengths_dev.unsqueeze(1)
        mask_f = mask.unsqueeze(-1).float()

        row_idx = torch.arange(B, device=x.device)
        last_idx = (lengths_dev - 1).clamp(min=0)
        last_hidden = lstm_out[row_idx, last_idx, :]

        mean_pool = (lstm_out * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)

        neg_inf = torch.finfo(lstm_out.dtype).min
        max_pool = lstm_out.masked_fill(~mask.unsqueeze(-1), neg_inf).max(dim=1).values

        row_idx  = torch.arange(B, device=x.device)
        last_idx = (lengths_dev - 1).clamp(min=0)
        current  = x[row_idx, last_idx, :]   # last REAL position (right-padded)
        stat_vec = self.static_mlp(current)
        feat = torch.cat([last_hidden, mean_pool, max_pool, stat_vec], dim=1)
        return self.head(feat).squeeze(-1)   # (B,)


In [77]:
# smoke test
N_FEATURES = X_tr.shape[2]
m = FraudLSTMCurrentTxn(N_FEATURES).to(device)
xb = torch.randn(8, MAX_LEN, N_FEATURES, device=device)
lb = torch.randint(1, MAX_LEN+1, (8,), device='cpu')
print('forward output shape:', m(xb, lb).shape)   # expect (8,)
del m, xb, lb


forward output shape: torch.Size([8])


## 8. Training loop (single UID-disjoint split, current-transaction prediction)

One model is trained on the ~80% train UIDs and evaluated on the held-out 20%.
Validation AUC is computed over one scalar prediction per validation transaction.


In [78]:
device = torch.device('cuda' if torch.cuda.is_available()
                       else 'mps' if torch.backends.mps.is_available()
                       else 'cpu')
print('device:', device)

# Hyperparameters for the current-transaction setup
BATCH               = 64
EPOCHS              = 30
LR                  = 1e-3
WEIGHT_DECAY        = 2e-4
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 6
HIDDEN_DIM          = 128
NUM_LAYERS          = 2
DROPOUT             = 0.3
BIDIRECTIONAL       = False
N_SEEDS             = 3


device: cuda


In [79]:
def scalar_bce(logits, targets):
    return nn.functional.binary_cross_entropy_with_logits(logits, targets)


In [80]:
def train_one_uid_run(X_tr, L_tr, y_tr,
                      X_va, L_va, y_va,
                      n_features, epochs, batch, lr, weight_decay,
                      device, patience, grad_clip, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FraudLSTMCurrentTxn(
        n_features=n_features,
        hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
        drop_prob=DROPOUT, bidirectional=BIDIRECTIONAL,
    ).to(device)

    train_loader = make_loader(X_tr, L_tr, y_tr, batch_size=batch, shuffle=True,  device=device)
    val_loader   = make_loader(X_va, L_va, y_va, batch_size=batch, shuffle=False, device=device)

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, epochs=epochs,
        steps_per_epoch=max(1, math.ceil(len(X_tr) / batch)),
        pct_start=0.1, anneal_strategy='cos',
    )

    best_auc, best_state, best_val_pred, bad = -1.0, None, None, 0
    for epoch in range(1, epochs + 1):
        model.train(); running, n_seen = 0.0, 0; t0 = time.time()
        for xb, lb, yb in train_loader:
            xb, lb = xb.to(device), lb.to(device)
            yb = yb.to(device)
            opt.zero_grad()
            logits = model(xb, lb)
            loss   = scalar_bce(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step(); sched.step()
            running += loss.item() * xb.size(0); n_seen += xb.size(0)
        train_loss = running / max(n_seen, 1)

        model.eval(); preds = []
        with torch.no_grad():
            for xb, lb, _ in val_loader:
                xb, lb = xb.to(device), lb.to(device)
                preds.append(torch.sigmoid(model(xb, lb)).cpu().numpy())
        val_pred = np.concatenate(preds)
        val_auc = roc_auc_score(y_va, val_pred)

        print(f'   ep {epoch:>2}/{epochs}  loss={train_loss:.4f}  '
              f'val_auc={val_auc:.4f}  ({time.time()-t0:.1f}s)')

        if val_auc > best_auc:
            best_auc      = val_auc
            best_state    = copy.deepcopy(model.state_dict())
            best_val_pred = val_pred
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f'   early stop at epoch {epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    return best_val_pred, best_auc, model


In [81]:
print(f'Current-transaction run with {N_SEEDS} seeds')
print(f'  train samples={X_tr.shape[0]:,}  val samples={X_va.shape[0]:,}')
print(f'  train UIDs={pd.Series(uids_tr).nunique():,}  val UIDs={pd.Series(uids_va).nunique():,}')

seed_val_preds, seed_aucs = [], []
SEED = 42
for s in range(N_SEEDS):
    seed = SEED + s
    print(f'\n-- seed {seed} --')
    val_p, val_auc, model = train_one_uid_run(
        X_tr, L_tr, y_tr,
        X_va, L_va, y_va,
        n_features=N_FEATURES,
        epochs=EPOCHS, batch=BATCH, lr=LR,
        weight_decay=WEIGHT_DECAY, device=device,
        patience=EARLY_STOP_PATIENCE, grad_clip=GRAD_CLIP, seed=seed,
    )
    seed_val_preds.append(val_p); seed_aucs.append(val_auc)
    del model
    if device.type == 'mps': torch.mps.empty_cache()
    gc.collect()

avg_val_pred = np.mean(seed_val_preds, axis=0)
overall_auc = roc_auc_score(y_va, avg_val_pred)
print(f'\n=== Per-seed AUCs: {[round(a,4) for a in seed_aucs]}')
print(f'=== Seed-averaged OOF AUC = {overall_auc:.4f} ===')


Current-transaction run with 3 seeds
  train samples=472,434  val samples=118,106
  train UIDs=175,043  val UIDs=42,692

-- seed 42 --
   ep  1/30  loss=0.1222  val_auc=0.8712  (79.6s)
   ep  2/30  loss=0.0907  val_auc=0.8762  (77.9s)
   ep  3/30  loss=0.0840  val_auc=0.8206  (78.1s)
   ep  4/30  loss=0.0758  val_auc=0.8270  (77.7s)
   ep  5/30  loss=0.0694  val_auc=0.8167  (77.9s)
   ep  6/30  loss=0.0646  val_auc=0.8252  (78.1s)
   ep  7/30  loss=0.0611  val_auc=0.8124  (78.1s)
   ep  8/30  loss=0.0582  val_auc=0.8123  (78.4s)
   early stop at epoch 8

-- seed 43 --
   ep  1/30  loss=0.1190  val_auc=0.8678  (76.9s)
   ep  2/30  loss=0.0901  val_auc=0.8572  (76.9s)
   ep  3/30  loss=0.0837  val_auc=0.8403  (77.0s)
   ep  4/30  loss=0.0758  val_auc=0.8106  (77.3s)
   ep  5/30  loss=0.0691  val_auc=0.8157  (77.9s)
   ep  6/30  loss=0.0646  val_auc=0.8250  (78.2s)
   ep  7/30  loss=0.0611  val_auc=0.8164  (77.7s)
   early stop at epoch 7

-- seed 44 --
   ep  1/30  loss=0.1192  val_auc=0

In [82]:
# Map current-transaction predictions back to validation TransactionIDs
assert len(orig_va) == len(avg_val_pred), \
    f'len mismatch: {len(orig_va)} vs {len(avg_val_pred)}'

oof_df = pd.DataFrame({
    'TransactionID': orig_va,
    'oof_lstm_current_txn': avg_val_pred,
    'isFraud_label': y_va,
})
# oof_df.to_csv('oof_lstm_current_txn_uid_disjoint.csv', index=False)
# print(f'Saved {len(oof_df):,} OOF rows to oof_lstm_current_txn_uid_disjoint.csv')
print(f'  overall AUC: {roc_auc_score(oof_df.isFraud_label, oof_df.oof_lstm_current_txn):.4f}')


  overall AUC: 0.8818


## 9. Save OOF predictions

v6 currently creates validation OOF predictions for the UID-disjoint split. Test
prediction is intentionally not wired here because this variant is for comparing
the scalar current-transaction objective against the v4 per-timestep objective.


In [ ]:
# Optional save:
# oof_df.to_csv('oof_lstm_current_txn_uid_disjoint.csv', index=False)
# print('Wrote oof_lstm_current_txn_uid_disjoint.csv')


## 10. Notes

**v6 summary**

- Starts from notebook v4 preprocessing and UID-disjoint validation.
- Changes the target shape from per-timestep `(batch, MAX_LEN)` to scalar `(batch,)`.
- Each sample is one current transaction plus that UID's available history up to the current row.
- Windows are right-padded and passed through `pack_padded_sequence`.
- Mean+max pooling is over the available history only; because the sequence ends at the current transaction, it does not include future transactions.
- There are no padding labels or label masks in v6. Each sequence has one `isFraud` target.

**Comparison target**

Use v6 to compare against v4 when you want to test whether scalar current-transaction prediction is better than per-timestep UID-sequence prediction under the same UID-disjoint validation idea.
